# Milestone - Unit 5: matrix factorization -- give every reader and every book a handful of hidden 'taste dials', learn them by gradient descent, and let one dot product do the recommending.

## 1. Load the catalog, the interactions, the keyword documents, and the cold readers

The same generated substrate Units 1-4 ran on. We load the keyword bags only so the
Unit-3 `LexicalRetrievalPath` (our content baseline) can be scored side by side with the
new latent-factor path. `bookrec.generated_dir()` reports a clear error if
`recsys/data/generated/` has not been built yet.

In [ ]:
import json
from collections import defaultdict

import bookrec
import numpy as np
import pandas as pd

K = 10
RANDOM_SEED = 0

data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")
catalog_ids = sorted(catalog)
keywords = bookrec.load_keywords(data_dir / "keywords.csv.gz")

cold = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = cold["cold_readers"]

interactions_path = data_dir / "interactions.csv.gz"
rows = pd.read_csv(interactions_path).to_dict("records")

train_positive = [r for r in rows if r["split"] == "train" and r["label"] == 1]
train_positive_items = {r["item_id"] for r in train_positive}
{
    "books": len(catalog_ids),
    "interactions": len(rows),
    "train_positives": len(train_positive),
    "books_with_a_train_positive": len(train_positive_items),
    "cold_books_zero_train_positives": len(catalog_ids) - len(train_positive_items),
    "cold_readers": len(cold_readers),
}

## 2. Train and register the matrix-factorization path beside the four earlier signals

`MatrixFactorizationPath` **learns** a low-dimensional factor (an embedding) for every
reader and every book, so that `sigmoid(P_u . Q_i)` approximates "reader u liked book
i". Where Unit 4's item-item CF reads co-occurrence straight off the log, MF trains `P`
and `Q` by **full-batch, per-entity-averaged gradient descent on a logistic
implicit-feedback loss** with L2 regularization. It trains on the **train positives**
(`split == 'train'` and `label == 1`) paired with `n_negatives` negatives sampled per
positive from each reader's *unobserved complement* -- NOT the log's exposure
`label == 0` rows, which carry the taste signal and would collapse MF to the random
floor. The config is **pinned** to the values measured on the shipped data
(`n_factors=32, n_epochs=300, learning_rate=0.5, reg=0.05, n_negatives=10, seed=0`),
because below ~300 epochs MF silently slips under the lexical path -- these are not
"small/few". One fit takes about 15 s. The registry now holds five signals: chance,
crowd counts, content words, who-read-what, and now learned latent factors.

In [ ]:
random_floor = bookrec.RandomRetrievalPath(catalog_ids, seed=RANDOM_SEED)
popularity = bookrec.PopularityRetrievalPath().fit(rows, catalog=catalog_ids)
lexical = bookrec.LexicalRetrievalPath(keywords).fit(rows, catalog=catalog_ids)
cf = bookrec.ItemItemRetrievalPath().fit(rows, catalog=catalog_ids)
mf = bookrec.MatrixFactorizationPath(seed=RANDOM_SEED).fit(rows, catalog=catalog_ids)  # ~15 s

registry = bookrec.PathRegistry()
for path in (random_floor, popularity, lexical, cf, mf):
    registry.register(path)

fitted = mf.artifact()
P = fitted["P"]
Q = fitted["Q"]
reader_ids = fitted["reader_ids"]
item_ids = fitted["item_ids"]
reader_of = {reader: idx for idx, reader in enumerate(reader_ids)}
row_of = {item_id: idx for idx, item_id in enumerate(item_ids)}
{
    "registered": [path.artifact_name() for path in registry],
    "P_shape_readers_x_factors": list(P.shape),
    "Q_shape_books_x_factors": list(Q.shape),
    "readers_with_a_learned_factor": len(reader_ids),
    "books_with_a_Q_row": len(item_ids),
}

## 3. Score all five paths on the frozen `val` scoreboard

The same scoreboard as Units 1-4: positive `val` rows, `k = 10`, the 60 cold readers
excluded, and a reader's `train` positives form the seen set so they are never
recommended back. Each path is graded by hit@10 -- the share of readers whose held-out
book lands in their top 10.

In [ ]:
boards = {
    path.name: bookrec.run_validation_scoreboard(
        path, interactions_path, catalog_ids=catalog_ids, k=K, cold_readers=cold_readers
    )
    for path in registry
}

floor_hit = boards["random"].hit_rate_at_k
pop_hit = boards["popularity"].hit_rate_at_k
lex_hit = boards["lexical"].hit_rate_at_k
cf_hit = boards["item-item"].hit_rate_at_k
mf_hit = boards["mf"].hit_rate_at_k

{
    "scored_readers": boards["mf"].readers,
    "random_hit_at_10": round(floor_hit, 4),
    "popularity_hit_at_10": round(pop_hit, 4),
    "lexical_hit_at_10": round(lex_hit, 4),
    "item_item_cf_hit_at_10": round(cf_hit, 4),
    "mf_hit_at_10": round(mf_hit, 4),
    "mf_over_popularity": round(mf_hit / pop_hit, 2),
    "mf_over_lexical": round(mf_hit / lex_hit, 2),
    "mf_minus_cf": round(mf_hit - cf_hit, 4),
}

Read the headline honestly. MF lands around 0.28 hit@10 here: roughly 2.5x the
popularity path (~0.108) and about 1.7x the lexical content path (~0.158), beating both
by wide margins. Against item-item CF (~0.252) it is **on par, not a reliable win**:
across negative-sampling seeds MF measures roughly 0.25-0.28, and this seed-0 run (a
touch above CF, see `mf_minus_cf`) sits inside the run-to-run noise over 500 readers, so
the honest story is parity. MF is the **latent generalization of co-occurrence**: it
does not out-score CF, it compresses the same who-read-what signal into a compact
32-number factor per reader and book -- and, unlike CF's all-zero cold column, it assigns
every catalog book a `Q` row, so it can at least *score* the whole catalog.

## 4. One reader's learned factor -> top MF recommendations (the score is a dot product)

Pick a reader deterministically (the lowest-id warm reader). Its learned factor
`P[reader]` is just a vector of `n_factors` numbers -- a point in a learned taste space.
MF scores a book by the **dot product** of that reader vector with the book's factor
vector `Q[book]`; the whole top-k is `P[reader] . Q^T` sorted and the seen books dropped.
We confirm below that each recommendation's score is exactly that dot product. These
learned reader/book vectors ARE embeddings, and a dot-product retriever over them is
precisely the shape Unit 8's neural two-tower will learn -- the Part-1 -> Part-2 hinge.

In [ ]:
seen_by = defaultdict(set)
for row in train_positive:
    if row["item_id"] in catalog:
        seen_by[row["reader_id"]].add(row["item_id"])

cold_set = set(cold_readers)
reader_id = min(r for r in reader_ids if r not in cold_set)
seen = seen_by[reader_id]
reader_factor = P[reader_of[reader_id]]

recommendations = mf.retrieve(reader_id, {"seen": seen}, K)

# The score MF ranks by IS the dot product of the reader embedding with each book embedding.
rebuilt = []
for candidate in recommendations:
    book = catalog[candidate.item_id]
    dot = float(reader_factor @ Q[row_of[candidate.item_id]])
    rebuilt.append(
        {
            "item_id": candidate.item_id,
            "title": book.title,
            "genres": book.fields["genres"],
            "mf_score": round(candidate.score, 3),
            "reader_factor_dot_book_factor": round(dot, 3),
        }
    )

print("reader", reader_id, "has", len(seen), "train positives")
print(
    "learned factor P[reader]: shape", reader_factor.shape,
    "| norm", round(float(np.linalg.norm(reader_factor)), 3),
)
print("first 8 of its", len(reader_factor), "taste dials:", reader_factor[:8].round(3))
pd.DataFrame(rebuilt)

Each recommendation's `mf_score` equals the `reader_factor . book_factor` dot product
beside it: the ranking is nothing more than a nearest-neighbor search in the learned
embedding space. That is the exact machinery Unit 8 keeps -- it swaps this hand-rolled
gradient descent for a neural two-tower that learns richer embeddings from the same
implicit feedback, but the final score stays a dot product of a reader vector and a book
vector.

## 5. The cold-item limit: a book with no train positive is shaped only by negative gradient

MF gives *every* catalog book a `Q` row, so it can score the whole catalog -- broader
reach than CF's hard-zero cold column. But 818 books have **zero** train positives. Such
a book is never a training positive; it only ever appears when it is *sampled as a
negative*, so gradient descent only ever pushes its factor in the direction that
**lowers** its score. We compare the MF scores our reader assigns to warm books (at least
one train positive somewhere) against the cold books (none), and we check what a cold
*reader* gets back.

In [ ]:
cold_items = [item_id for item_id in item_ids if item_id not in train_positive_items]
warm_items = [item_id for item_id in item_ids if item_id in train_positive_items]

scores = reader_factor @ Q.T  # this reader's MF score for every catalog book
cold_scores = np.array([scores[row_of[i]] for i in cold_items])
warm_scores = np.array([scores[row_of[i]] for i in warm_items])

ranked = scores.argsort()[::-1]  # best-scoring books first
ranked_ids = [item_ids[idx] for idx in ranked]
best_cold_rank = next(
    position
    for position, item_id in enumerate(ranked_ids, start=1)
    if item_id not in train_positive_items
)

# A cold READER (0 train positives) has no learned P row at all -> retrieve returns [].
cold_reader = cold_readers[0]
cold_reader_recs = mf.retrieve(cold_reader, {"seen": set()}, K)

{
    "cold_books_scored": len(cold_items),
    "warm_books_scored": len(warm_items),
    "mean_mf_score_cold_books": round(float(cold_scores.mean()), 3),
    "mean_mf_score_warm_books": round(float(warm_scores.mean()), 3),
    "top_100_are_all_warm": all(i in train_positive_items for i in ranked_ids[:100]),
    "best_cold_book_rank_out_of_2000": best_cold_rank,
    "cold_reader_example": cold_reader,
    "recs_for_a_cold_reader": len(cold_reader_recs),
}

So MF has two very different "cold" stories, and this milestone shows both. A cold
*reader* (0 train positives) gets **no** `P` row at all, so `retrieve` returns `[]`
(`recs_for_a_cold_reader` is 0) -- there is simply no taste vector to score with. A cold
*book* does get a `Q` row, but with no positive signal its factor is dragged down by
negative sampling, so it scores well below the warm books and never surfaces in the top
ranks (the best cold book sits far down the list). Id-only matrix factorization cannot
invent taste for a book no training reader engaged; closing that coverage gap is the job
of Unit 6's blending and Unit 9's feature-based item cold start. What MF *does* deliver
is the embedding bridge -- reader and book vectors whose dot product is the score -- the
foundation the neural two-tower of Unit 8 builds on.